# Notebook 01: Preprocesamiento e Ingesta del Dataset

## Contexto y Objetivo
Este notebook documenta la primera fase del pipeline de datos para el sistema **ClaimVision-AI**.
El objetivo es transformar el dataset crudo de imágenes de siniestros vehiculares y sus correspondientes anotaciones en formato COCO JSON hacia estructuras tabulares limpias y normalizadas (CSVs) para el entrenamiento de modelos de visión.

### Etapas:
1. Lectura y exploración de metadatos de imágenes vehiculares.
2. Normalización de rutas y conversión de anotaciones espaciales a DataFrames de pandas.
3. Validación de integridad de imágenes y particionado inicial (Train / Val / Test).


In [2]:
import cv2 as cv
import os
import matplotlib.pyplot as plt
import config
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import pandas as pd

In [3]:
raw_images_dir = F"{config.RAW_PATH_ORIGINAL}/car-images"
processed_images_dir = F"{config.PROCESSED_PATH_ORIGINAL}/car-images"
train_csv_path = F"{config.RAW_PATH_ORIGINAL}/csv/dev.csv"
val_csv_path = F"{config.RAW_PATH_ORIGINAL}/csv/val.csv"

### Función de Homogeneización Espacial (Square Padding)
Para evitar distorsión de aspecto (*aspect ratio*) al redimensionar las fotografías de siniestros, aplicamos una transformación de relleno (*padding*) dinámico para convertir cada imagen a un formato cuadrado antes de estandarizar su resolución.

In [5]:
def make_square_with_padding_dynamic(image_path, size=512):
    imagen = Image.open(image_path).convert('RGB')
    image_arr = np.array(imagen)
    prom_color = image_arr.mean(axis=(0, 1)).astype(int)
    background = tuple(prom_color)
    width, height = imagen.size

    if width > height:
        new_height = int(size * (height / width))
        new_width = size
    else:
        new_width = int(size * (width / height))
        new_height = size

    resized_image = imagen.resize((new_width, new_height), Image.LANCZOS)
    square_image = Image.new('RGB', (size, size), background)
    pos = ((size - new_width) // 2, (size - new_height) // 2)
    square_image.paste(resized_image, pos)

    return square_image

## Primero movemos los train y test 

In [83]:
df = pd.read_csv(train_csv_path)

### Particionado Estratificado del Dataset
Separamos el conjunto de entrenamiento (80%) y prueba (20%) fijando la semilla aleatoria para garantizar reproducibilidad en las evaluaciones.

In [84]:
train_set, test_set = train_test_split(df, train_size=0.8,random_state=42)

# Movemos los elementos de una carpeta a otra haciendo el resize.
Primero Train, luego Test. Renombramos cada imagen y ajustamos su ruta destino

In [86]:
i = 0
for index, row in train_set.iterrows():
    image_path = row['image_path']
    source_path = os.path.join(F"{raw_images_dir}/images/", image_path)    
    destination_path = os.path.join(F"{processed_images_dir}/train/", F"{i}.jpg")
    
    new_image = make_square_with_padding_dynamic(source_path, 512)
    train_set.at[index, 'image_path'] = destination_path
    new_image.save(destination_path)
    
    i = i+1

In [87]:
i = 0
for index, row in test_set.iterrows():
    image_path = row['image_path']
    source_path = os.path.join(F"{raw_images_dir}/images/", image_path)    
    destination_path = os.path.join(F"{processed_images_dir}/test/", F"{i}.jpg")
    
    new_image = make_square_with_padding_dynamic(source_path, 512)
    test_set.at[index, 'image_path'] = destination_path
    new_image.save(destination_path)
    
    i = i+1

In [32]:
df = pd.read_csv(val_csv_path)
val_set = df["image_path"].copy()
val_set.to_csv(F"{config.PROCESSED_PATH_ORIGINAL}/csv/val.csv", index=False)

In [34]:
df = pd.read_csv("./data/processed/original/csv/val.csv")
val_set = df

In [39]:
i = 0
for index, row in val_set.iterrows():
    image_path = row['image_path']
    source_path = os.path.join(F"{raw_images_dir}/val/", image_path)    
    destination_path = os.path.join(F"{processed_images_dir}/val/", F"{i}.jpg")
    
    new_image = make_square_with_padding_dynamic(source_path, 512)
    val_set.at[index, 'image_path'] = destination_path
    new_image.save(destination_path)
    
    i = i+1

In [40]:
val_set.to_csv(F"{config.PROCESSED_PATH_ORIGINAL}/csv/val.csv", index=False)

Actualizamos y guardamos los CSV en la carpeta correspondiente

In [88]:
train_set.to_csv(F"{config.PROCESSED_PATH_ORIGINAL}/csv/train.csv", index=False)
test_set.to_csv(F"{config.PROCESSED_PATH_ORIGINAL}/csv/test.csv", index=False)

## Ejemplo

In [31]:
example_image_dir =  F"{config.PROYECT_DIR}/test_images"

In [32]:
example_image_path_load = F"{example_image_dir}/image1.jpg"
image_resultante = make_square_with_padding_dynamic(example_image_path_load)
image_resultante.show()

In [33]:
example_image_path_save = F"{example_image_dir}/save_image1.jpg"
image_resultante.save(example_image_path_save)